# ES-LLM: Layer-wise Fine-Tuning mit Evolutionary Strategies

Dieses Notebook führt dich durch den kompletten Workflow:
1. **Setup** – Repo klonen, Dependencies installieren
2. **Inspect** – Modell-Architektur inspizieren
3. **Baseline** – Unveränderte Accuracy messen
4. **Train** – ES-Training auf ausgewählten Layern
5. **Evaluate** – Fine-tuned Modell testen
6. **Export** – Ergebnisse nach Google Drive speichern

**GPU-Empfehlung:** Runtime → Change runtime type → **T4 GPU** (kostenlos) oder **A100** (Colab Pro)

## 0. GPU prüfen

In [ ]:
import torch
print(f"CUDA verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Keine GPU! Gehe zu Runtime → Change runtime type → GPU")

## 1. Setup – Repo klonen & Dependencies installieren

In [ ]:
# ── Repo klonen ──
import os

REPO_URL = "https://github.com/Haso001/PG_ML_AI.git"
REPO_DIR = "/content/PG_ML_AI"
PROJECT_DIR = f"{REPO_DIR}/es-llm-finetune-Hasan-Dev"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Repo existiert → neueste Änderungen pullen
    !cd {REPO_DIR} && git pull
    print(f"{REPO_DIR} aktualisiert (git pull).")

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Alternativ: Repo als ZIP hochladen ──
# Falls du kein Git-Repo hast, lade den Ordner als ZIP hoch:
#
# from google.colab import files
# uploaded = files.upload()  # ZIP-Datei hochladen
# !unzip -o es-llm-finetune.zip -d /content/es-llm-finetune
# os.chdir("/content/es-llm-finetune")

In [ ]:
# ── Dependencies installieren ──
!pip install -q torch transformers datasets accelerate pyyaml cmaes

# Verifizieren
import torch, transformers
print(f"torch={torch.__version__}, transformers={transformers.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# ── src/ zum Python-Pfad hinzufügen ──
import sys
sys.path.insert(0, f"{PROJECT_DIR}/src")

# Import-Test
from es_llm.model.loader import load_model_and_tokenizer, inspect_model_layers
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k import GSM8KFitness
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness
from es_llm.training.es_trainer import train
from es_llm.utils.config import load_config
print("Alle Imports erfolgreich!")

## 2. Modell-Architektur inspizieren

Zuerst schauen wir uns an, welche Layer das Qwen2.5-0.5B hat und wie viele Parameter pro Layer existieren.

In [ ]:
# Modell laden (float16 auf GPU → ~1 GB VRAM)
model, tokenizer = load_model_and_tokenizer(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    dtype="float16",
    device="cuda",
)

total = sum(p.numel() for p in model.parameters())
print(f"\nGesamt: {total:,} Parameter ({total/1e6:.1f}M)")
print(f"VRAM belegt: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Alle Layer auflisten
layers = inspect_model_layers(model)

# Per-Layer Zusammenfassung
from collections import defaultdict
per_layer = defaultdict(int)
for l in layers:
    parts = l["name"].split(".")
    key = "other"
    for i, p in enumerate(parts):
        if p == "layers" and i + 1 < len(parts) and parts[i + 1].isdigit():
            key = f"layers.{parts[i + 1]}"
            break
    per_layer[key] += l["numel"]

print(f"{'Layer':<20s} {'Parameter':>15s}")
print("─" * 37)
for key in sorted(per_layer, key=lambda k: (not k.startswith("layers"), k)):
    print(f"{key:<20s} {per_layer[key]:>15,}")

In [ ]:
# Detail: Layer 23 (letzter Decoder-Block)
print("\nLayer 23 – Detail:")
print(f"{'Name':<55s} {'Shape':<25s} {'Params':>10s}")
print("─" * 92)
for l in layers:
    if "layers.23" in l["name"]:
        print(f"{l['name']:<55s} {str(l['shape']):<25s} {l['numel']:>10,}")

## 3. Layer-Selektion testen

Wir nutzen den **kompletten letzten Decoder-Block** (Layer 23) mit allen Komponenten:
- **Attention**: q_proj, k_proj, v_proj, o_proj
- **MLP**: gate_proj, up_proj, down_proj
- **LayerNorm**: input_layernorm, post_attention_layernorm

Das ergibt **~14.9M Parameter** als einen einzigen Vektor θ, den ES optimiert.

In [ ]:
# Kompletter letzter Layer → alle ~14.9M Parameter (Attention + MLP + LayerNorm)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[23],
    components="all",
)
print(selector.summary())
print(f"\nGesamt: {selector.num_target_elements:,} Parameter als 1-D Vektor θ")

In [ ]:
# MLP des letzten Layers → mehr Parameter
selector_mlp = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[23],
    components="mlp",
)
print(selector_mlp.summary())

In [ ]:
# Attention Q/K/V der letzten 2 Layer
selector_attn = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[22, 23],
    components="attention_qkv",
)
print(selector_attn.summary())

## 4. Baseline messen

Wie gut ist das Modell **unverändert** auf GSM8K?

In [ ]:
# Baseline-Fitness (kleines Subset zum schnellen Testen)
fitness_fn = GSM8KFitness(
    num_samples=30,       # Anzahl Fragen
    split="test",
    max_new_tokens=256,
)

baseline_acc = fitness_fn.evaluate(model, tokenizer)
print(f"\n✅ Baseline Accuracy (GSM8K test, n=30): {baseline_acc:.2%}")

## 5. ES-Training starten

### Option A: Mit YAML-Config (empfohlen)

In [ ]:
# Config laden und starten
cfg = load_config(f"{PROJECT_DIR}/configs/gsm8k_last_layer.yaml")

# Config anzeigen
import json
print(json.dumps(cfg, indent=2))

In [ ]:
# Optional: Werte überschreiben für schnellen Test mit vollem Layer
cfg["layer_selection"]["layer_indices"] = [23]
cfg["layer_selection"]["components"] = "all"   # alle ~14.9M Params

cfg["es"]["num_generations"] = 10              # Schneller Testlauf
cfg["es"]["population_size"] = 10
cfg["es"]["sigma"] = 0.001                     # Klein weil 14.9M Dimensionen
cfg["es"]["learning_rate"] = 0.001

cfg["fitness"]["num_samples"] = 15             # Samples pro Fitness-Eval
cfg["fitness"]["max_new_tokens"] = 128

cfg["output"]["dir"] = "/content/experiments/runs"

# ACHTUNG: Bei Option A wird das Modell NEU geladen.
# Vorheriges Modell freigeben, um VRAM zu sparen:
del model, tokenizer
torch.cuda.empty_cache()

# Training starten!
run_dir = train(cfg)
print(f"\n✅ Training abgeschlossen! Ergebnisse in: {run_dir}")

### Option B: Manueller Training-Loop (mehr Kontrolle)

Zwei Varianten:
- **B1: Accuracy-Fitness** – generiert Antworten, prüft ob richtig (langsam, diskret)
- **B2: Log-Likelihood-Fitness** – misst Wahrscheinlichkeit der richtigen Antwort (schnell, stetig)

**Empfehlung: B2 (Log-Likelihood)** → ~10× schneller, besseres Signal für ES

In [ ]:
# ── Option B1: Manueller ES-Loop mit ACCURACY-Fitness (langsam, diskret) ──
# (Gleicher Ansatz wie der erste Run → ~10 Min./Generation)

import torch
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k import GSM8KFitness

# 1) Modell laden
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswählen – KOMPLETTER letzter Layer (~14.9M Parameter)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[23],       # Letzter Decoder-Block
    components="all",         # Attention + MLP + LayerNorm → ~14.9M Params
)
print(f"Target: {selector.num_target_elements:,} Parameter")

# 3) ES-Algorithmus
es = OpenAIES(
    population_size=200,
    sigma=0.001,
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
)

# 4) Fitness-Funktion (Accuracy – langsam)
fitness_fn = GSM8KFitness(num_samples=15, split="train", max_new_tokens=128)

# 5) Training
current_params = selector.get_flat_params().clone()
history = []

NUM_GENERATIONS = 10

for gen in range(1, NUM_GENERATIONS + 1):
    candidates = es.ask(current_params)
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)
        torch.cuda.empty_cache()
    
    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)
    
    history.append({"gen": gen, "best": result.best_fitness, "mean": result.mean_fitness})
    print(f"Gen {gen:3d} | best={result.best_fitness:.3f}  mean={result.mean_fitness:.3f}")

print(f"\n✅ Training abgeschlossen nach {NUM_GENERATIONS} Generationen")

In [ ]:
# ── Option B2: Manueller ES-Loop mit LOG-LIKELIHOOD-Fitness (schnell, stetig) ──
# ~10x schneller als B1 → ~1 Min./Generation statt ~10 Min.

import torch
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness

# 1) Modell laden
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswaehlen - KOMPLETTER letzter Layer (~14.9M Parameter)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[23],
    components="all",
)
print(f"Target: {selector.num_target_elements:,} Parameter")

# 3) ES-Algorithmus (gleiche Settings wie erster Run)
es = OpenAIES(
    population_size=10,
    sigma=0.001,
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
)

# 4) Fitness-Funktion (Log-Likelihood - schnell und stetig!)
fitness_fn = GSM8KLogLikelihoodFitness(
    num_samples=30,        # mehr Samples moeglich weil viel schneller
    split="train",
    target_mode="short",   # nur "#### <number>" bewerten
)

# 5) Training
current_params = selector.get_flat_params().clone()
history = []

NUM_GENERATIONS = 10

for gen in range(1, NUM_GENERATIONS + 1):
    candidates = es.ask(current_params)
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)
        torch.cuda.empty_cache()
    
    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)
    
    history.append({"gen": gen, "best": result.best_fitness, "mean": result.mean_fitness})
    print(f"Gen {gen:3d} | best={result.best_fitness:.4f}  mean={result.mean_fitness:.4f}")

print(f"\nLL-Training abgeschlossen nach {NUM_GENERATIONS} Generationen")

### Option C: Optimiertes ES-Training v4 für A100/H100 (empfohlen)

Basierend auf **Salimans et al. 2017** – "Evolution Strategies as a Scalable Alternative to RL"

**v4-Verbesserung gegenüber v3: Adaptive Sigma**

| Problem in v3 | Fix in v4 |
|---|---|
| **Cosine σ-Decay**: blind, ignoriert Training-Dynamik | **Adaptive σ**: reagiert auf Fitness-Verlauf |
| σ fällt auch bei Stagnation → kein Entkommen aus Plateau | Stagnation (5 Gen) → σ wächst (*1.5) → neue Exploration |
| σ kann unnötig hoch bleiben wenn Training gut läuft | Verbesserung → σ schrumpft (*0.85) → feinere Suche |

**Kernfeatures:**
- **Adaptive Sigma**: σ ∈ [0.001, 0.01], steigt bei Stagnation, fällt bei Verbesserung
- **Full-Reasoning Target**: LL über gesamte Step-by-Step Lösung
- **Data Reshuffling per Generation**: 100 Fragen/Gen aus Pool von 5000
- **Adam-Optimizer**
- **Mittlere Layers [20,21]** → kein Catastrophic Forgetting

**Daten-Trennung:** Training auf `train`-Split (7473), Evaluation auf `test`-Split (1319)

**Erwartete Laufzeit:** ~30-45 Min. auf A100

In [ ]:
# ── Option C: Optimiertes ES-Training v4 – Adaptive Sigma ──
# Sigma reagiert auf Training-Dynamik: steigt bei Stagnation, fällt bei Verbesserung
# Budget: ~30-45 Min. auf A100

import torch, time
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness

# 1) Modell laden
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswählen – Attention von Layer 20+21 (~3.6M Parameter)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[20, 21],
    components="attention",
)
print(f"Target: {selector.num_target_elements:,} Parameter")
print(selector.summary())

# 3) ES mit Adam-Optimizer + ADAPTIVE Sigma
NUM_GENERATIONS = 80

es = OpenAIES(
    population_size=96,
    sigma=0.004,                      # Start-Sigma
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
    weight_decay=0.003,
    optimizer="adam",
    # Adaptive Sigma: reagiert auf Training-Dynamik
    sigma_decay="adaptive",
    sigma_adapt_patience=5,           # Nach 5 Gen Stagnation → σ hoch
    sigma_adapt_up=1.5,               # Stagnation: σ * 1.5
    sigma_adapt_down=0.85,            # Verbesserung: σ * 0.85
    sigma_min=0.001,                  # Nie unter 0.001
    sigma_max=0.01,                   # Nie über 0.01
    max_generations=NUM_GENERATIONS,
)

# 4) Fitness: Full-Reasoning LL mit per-Generation Reshuffling
fitness_fn = GSM8KLogLikelihoodFitness(
    num_samples=100,
    split="train",
    target_mode="full",
    batch_size=8,
    reshuffle=True,
    pool_size=5000,
)

# 5) Training Loop
current_params = selector.get_flat_params().clone()
best_ever = -float("inf")
best_ever_params = current_params.clone()
history = []

print(f"\n{'='*65}")
print(f"  ES-Training v4: {NUM_GENERATIONS} Generationen, Pop={es.population_size}")
print(f"  Optimizer: Adam (β1={es.adam_beta1}, β2={es.adam_beta2})")
print(f"  σ-Strategie: ADAPTIVE (patience=5, up=1.5x, down=0.85x)")
print(f"  σ-Bereich: [{es.sigma_min:.4f}, {es.sigma_max:.4f}]")
print(f"  Target: {selector.num_target_elements:,} Parameter (Layer 20+21 Attention)")
print(f"  Fitness: Full-Reasoning LL (n={fitness_fn.num_samples}, batch={fitness_fn.batch_size})")
print(f"  Data: Reshuffle per GENERATION – 100 gleiche Fragen für alle Kandidaten")
print(f"  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB belegt")
print(f"{'='*65}\n")

t_total = time.time()

for gen in range(1, NUM_GENERATIONS + 1):
    t0 = time.time()

    # Neue Fragen für diese Generation (alle Kandidaten sehen die gleichen!)
    fitness_fn.reshuffle_data()

    candidates = es.ask(current_params)
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)

    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)

    if result.best_fitness > best_ever:
        best_ever = result.best_fitness
        best_ever_params = current_params.clone()

    elapsed = time.time() - t0
    history.append({
        "gen": gen, "best": result.best_fitness, "mean": result.mean_fitness,
        "std": result.std_fitness, "best_ever": best_ever,
        "sigma": es.current_sigma, "elapsed": elapsed,
    })

    if gen % 10 == 0 or gen == 1:
        total_t = time.time() - t_total
        eta = (total_t / gen) * (NUM_GENERATIONS - gen)
        print(f"Gen {gen:3d}/{NUM_GENERATIONS} | best={result.best_fitness:.4f}  "
              f"mean={result.mean_fitness:.4f}  best_ever={best_ever:.4f}  "
              f"σ={es.current_sigma:.5f}  [{elapsed:.1f}s/gen, {total_t:.0f}s total, "
              f"ETA ~{eta/60:.0f}min]")

    torch.cuda.empty_cache()

# Best-ever Parameter anwenden
selector.set_flat_params(best_ever_params)
total_time = time.time() - t_total
print(f"\n✅ Training abgeschlossen in {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"   Best-ever LL Fitness: {best_ever:.4f}")

In [ ]:
# ── Vergleich: Fine-tuned (Option C v4) vs Base-Modell auf GSM8K TEST-Split ──
# WICHTIG: Evaluation auf TEST-Split (1319 Fragen) – komplett andere Daten als Training!
# Training nutzte TRAIN-Split (7473 Fragen) → kein Data Leakage

from es_llm.fitness.gsm8k import GSM8KFitness
from es_llm.model.loader import load_model_and_tokenizer as load_base
import torch, time

NUM_EVAL = 200  # 200 Fragen aus dem TEST-Split

# --- 1) Fine-tuned Modell evaluieren (aktuell im Speicher) ---
print(f"Evaluiere FINE-TUNED Modell (Option C v4) auf {NUM_EVAL} GSM8K-TEST-Fragen...")
print(f"  (Training war auf TRAIN-Split, Eval ist auf TEST-Split → faire Bewertung)")
eval_fn = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
finetuned_acc = eval_fn.evaluate(model, tokenizer)
t_ft = time.time() - t0
print(f"  Fine-tuned Accuracy: {finetuned_acc:.2%}  ({t_ft:.0f}s)")

# --- 2) Base-Modell laden und evaluieren ---
print(f"\nLade Base-Modell für Vergleich...")
base_model, base_tokenizer = load_base(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

print(f"Evaluiere BASE-Modell auf {NUM_EVAL} GSM8K-TEST-Fragen...")
eval_fn_base = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
base_acc = eval_fn_base.evaluate(base_model, base_tokenizer)
t_base = time.time() - t0
print(f"  Base Accuracy:       {base_acc:.2%}  ({t_base:.0f}s)")

# Base-Modell freigeben
del base_model, base_tokenizer
torch.cuda.empty_cache()

# --- 3) Ergebnis ---
diff = finetuned_acc - base_acc
print(f"\n{'='*60}")
print(f"  EVALUATION AUF GSM8K TEST-SPLIT (n={NUM_EVAL})")
print(f"  (Training war auf TRAIN-Split → kein Data Leakage)")
print(f"{'='*60}")
print(f"  Base-Modell:      {base_acc:6.2%}  ({int(base_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Fine-tuned (v4):  {finetuned_acc:6.2%}  ({int(finetuned_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Differenz:        {diff:+6.2%}")
print(f"{'='*60}")

In [ ]:
# ── Visualisierung: Training + Accuracy-Vergleich ──

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

# ── Plot 1: Fitness über Generationen ──
ax1 = fig.add_subplot(gs[0, 0])
gens = [h["gen"] for h in history]
bests = [h["best"] for h in history]
means = [h["mean"] for h in history]
best_evers = [h["best_ever"] for h in history]

ax1.plot(gens, bests, label="Best Fitness", linewidth=1.5, alpha=0.7, color="#2196F3")
ax1.plot(gens, means, label="Mean Fitness", linewidth=1.5, alpha=0.5, color="#FF9800")
ax1.plot(gens, best_evers, label="Best-Ever", linewidth=2, color="#4CAF50", linestyle="--")
ax1.set_xlabel("Generation")
ax1.set_ylabel("Log-Likelihood Fitness")
ax1.set_title("ES Training v4 – Fitness über Generationen\n(Adaptive σ + Data Reshuffling)")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Adaptive Sigma-Verlauf ──
ax2 = fig.add_subplot(gs[0, 1])
sigmas = [h["sigma"] for h in history]
ax2.plot(gens, sigmas, linewidth=2, color="#9C27B0")
ax2.set_xlabel("Generation")
ax2.set_ylabel("σ (Noise Std)")
ax2.set_title("Adaptive σ-Verlauf\n(↑ bei Stagnation, ↓ bei Verbesserung)")
ax2.grid(True, alpha=0.3)
ax2.fill_between(gens, sigmas, alpha=0.15, color="#9C27B0")
ax2.axhline(y=0.001, color="red", linestyle=":", alpha=0.4, label="σ_min")
ax2.axhline(y=0.01, color="red", linestyle=":", alpha=0.4, label="σ_max")
ax2.legend(fontsize=8)

# ── Plot 3: Accuracy-Vergleich (Balkendiagramm) ──
ax3 = fig.add_subplot(gs[1, 0])
labels = ["Base-Modell\n(unverändert)", "Fine-tuned v4\n(ES + Adaptive σ)"]
accs = [base_acc, finetuned_acc]
colors = ["#4C72B0", "#55A868"]

bars = ax3.bar(labels, accs, color=colors, width=0.5, edgecolor="black", linewidth=0.8)
for bar, acc in zip(bars, accs):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f"{acc:.1%}\n({int(acc*NUM_EVAL)}/{NUM_EVAL})",
             ha="center", va="bottom", fontweight="bold", fontsize=11)

ax3.set_ylabel("Accuracy")
ax3.set_title(f"GSM8K TEST Accuracy (n={NUM_EVAL})\nTrain auf train-Split, Eval auf test-Split")
ax3.set_ylim(0, max(accs) * 1.3)
ax3.axhline(y=base_acc, color="red", linestyle="--", alpha=0.3, label="Baseline")
ax3.grid(axis="y", alpha=0.3)
ax3.legend(fontsize=9)

# ── Plot 4: Zeitverlauf pro Generation ──
ax4 = fig.add_subplot(gs[1, 1])
times = [h["elapsed"] for h in history]
ax4.plot(gens, times, linewidth=1.5, color="#FF5722", alpha=0.7)
ax4.axhline(y=np.mean(times), color="gray", linestyle="--", alpha=0.5,
            label=f"Ø {np.mean(times):.1f}s/gen")
ax4.set_xlabel("Generation")
ax4.set_ylabel("Zeit (Sekunden)")
ax4.set_title("Laufzeit pro Generation")
ax4.grid(True, alpha=0.3)
ax4.legend(fontsize=9)

fig.suptitle("ES-Training v4 – Adaptive Sigma + Full Reasoning + Layer 20/21",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Zusammenfassung ──
total_time = sum(times)
print(f"\n{'='*65}")
print(f"  ZUSAMMENFASSUNG – ES-Training v4 (Adaptive Sigma)")
print(f"{'='*65}")
print(f"  Generationen:      {len(history)}")
print(f"  Population:        {es.population_size}")
print(f"  Parameter:         {selector.num_target_elements:,} (Layer 20+21 Attention)")
print(f"  Gesamtzeit:        {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"  Ø pro Generation:  {np.mean(times):.1f}s")
print(f"  σ Start → Ende:   {history[0]['sigma']:.5f} → {history[-1]['sigma']:.5f}")
print(f"  σ Min/Max:         {min(sigmas):.5f} / {max(sigmas):.5f}")
print(f"  Best-Ever LL:      {history[-1]['best_ever']:.4f}")
print(f"  Data Reshuffling:  ON (100 neue Fragen/Gen aus Pool von 5000)")
print(f"  Target Mode:       full (gesamtes Reasoning)")

In [ ]:
# ── Vergleich: Fine-tuned vs Base-Modell auf GSM8K (Accuracy) ──
# Testet das gerade trainierte Modell gegen das unveränderte Base-Modell
# mit jeweils 100 Fragen aus dem TEST-Split.

from es_llm.fitness.gsm8k import GSM8KFitness
import torch, time
import matplotlib.pyplot as plt

NUM_EVAL = 100

# --- 1) Fine-tuned Modell evaluieren (aktuell im Speicher) ---
print(f"Evaluiere FINE-TUNED Modell auf {NUM_EVAL} GSM8K-Testfragen...")
eval_fn = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
finetuned_acc = eval_fn.evaluate(model, tokenizer)
t_ft = time.time() - t0
print(f"  Fine-tuned Accuracy: {finetuned_acc:.2%}  ({t_ft:.0f}s)")

# --- 2) Base-Modell laden und evaluieren ---
print(f"\nLade Base-Modell fuer Vergleich...")
from es_llm.model.loader import load_model_and_tokenizer as load_base
base_model, base_tokenizer = load_base(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

print(f"Evaluiere BASE-Modell auf {NUM_EVAL} GSM8K-Testfragen...")
eval_fn_base = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
base_acc = eval_fn_base.evaluate(base_model, base_tokenizer)
t_base = time.time() - t0
print(f"  Base Accuracy:       {base_acc:.2%}  ({t_base:.0f}s)")

# Base-Modell freigeben
del base_model, base_tokenizer
torch.cuda.empty_cache()

# --- 3) Ergebnis ---
diff = finetuned_acc - base_acc
print(f"\n{'='*50}")
print(f"  Base-Modell:     {base_acc:6.2%}  ({int(base_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Fine-tuned:      {finetuned_acc:6.2%}  ({int(finetuned_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Differenz:       {diff:+6.2%}")
print(f"{'='*50}")

# --- 4) Plot ---
labels = ["Base-Modell", "Fine-tuned\n(ES + LL)"]
accs = [base_acc, finetuned_acc]
colors = ["#4C72B0", "#55A868"]

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(labels, accs, color=colors, width=0.5, edgecolor="black", linewidth=0.8)

# Werte auf die Balken schreiben
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{acc:.1%}\n({int(acc*NUM_EVAL)}/{NUM_EVAL})",
            ha="center", va="bottom", fontweight="bold", fontsize=12)

ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title(f"GSM8K Accuracy – Base vs Fine-tuned (n={NUM_EVAL})", fontsize=13)
ax.set_ylim(0, max(accs) * 1.25)
ax.axhline(y=base_acc, color="red", linestyle="--", alpha=0.4, label="Baseline")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Ergebnisse visualisieren

In [ ]:
import matplotlib.pyplot as plt
import json

# Falls history nicht im Speicher ist (z.B. nach Kernel-Neustart):
# → Aus training_log.jsonl laden (Option A) oder manuell definieren
if "history" not in dir():
    # Option A: Aus JSONL-Datei laden
    from pathlib import Path
    log_path = Path("/content/experiments/runs")
    # Neuesten Run finden
    if log_path.exists():
        runs = sorted(log_path.iterdir(), key=lambda p: p.name, reverse=True)
        for run in runs:
            jsonl = run / "training_log.jsonl"
            if jsonl.exists():
                with open(jsonl) as f:
                    history = [json.loads(l) for l in f]
                print(f"History geladen aus: {jsonl}")
                break
    if "history" not in dir():
        raise RuntimeError(
            "Variable 'history' nicht gefunden!\n"
            "Bitte zuerst eine Trainings-Zelle ausfuehren (Option A, B1 oder B2)."
        )

gens = [h.get("gen") or h.get("generation") for h in history]
bests = [h.get("best") or h.get("best_fitness") for h in history]
means = [h.get("mean") or h.get("mean_fitness") for h in history]

plt.figure(figsize=(10, 5))
plt.plot(gens, bests, label="Best Fitness", linewidth=2)
plt.plot(gens, means, label="Mean Fitness", alpha=0.7)
if "baseline_acc" in dir():
    plt.axhline(y=baseline_acc, color="red", linestyle="--", label="Baseline")
plt.xlabel("Generation")
plt.ylabel("Fitness")
plt.title("ES Training – Fitness über Generationen")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Fine-tuned Modell evaluieren

In [ ]:
# Evaluation auf dem Test-Split (größeres Subset)
eval_fitness = GSM8KFitness(
    num_samples=100,
    split="test",
    max_new_tokens=256,
)

final_acc = eval_fitness.evaluate(model, tokenizer)
print(f"\n📊 Ergebnisse:")
print(f"   Baseline (test, n=30):   {baseline_acc:.2%}")
print(f"   Fine-tuned (test, n=100): {final_acc:.2%}")
print(f"   Differenz:                {final_acc - baseline_acc:+.2%}")

## 8. Modell & Ergebnisse speichern

In [ ]:
# ── Checkpoint speichern ──
import json
from pathlib import Path

save_dir = Path("/content/experiments/final")
save_dir.mkdir(parents=True, exist_ok=True)

# Layer-Gewichte speichern
layer_state = {name: p.data.cpu().clone() for name, p in selector.get_target_params()}
torch.save(layer_state, save_dir / "best_layer_weights.pt")

# Komplettes Modell speichern
model.save_pretrained(save_dir / "model")
tokenizer.save_pretrained(save_dir / "model")

# Ergebnisse speichern
results = {
    "baseline_accuracy": baseline_acc,
    "final_accuracy": final_acc,
    "target_layers": selector.target_names,
    "num_target_params": selector.num_target_elements,
    "history": history,
}
(save_dir / "results.json").write_text(json.dumps(results, indent=2, default=str))

print(f"✅ Gespeichert in: {save_dir}")
!ls -la {save_dir}

In [ ]:
# ── Optional: Nach Google Drive kopieren ──
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/es-llm-experiments"
!mkdir -p {DRIVE_DIR}
!cp -r {save_dir}/* {DRIVE_DIR}/
print(f"✅ Kopiert nach Google Drive: {DRIVE_DIR}")

## 9. Gespeichertes Modell laden (in neuer Session)

So lädst du ein zuvor gespeichertes ES-Modell:

In [ ]:
# ── Variante A: Komplettes gespeichertes Modell ──
# model, tokenizer = load_model_and_tokenizer(
#     model_name="/content/experiments/final/model",
#     dtype="float16",
#     device="cuda",
# )

# ── Variante B: Base-Modell + Layer-Gewichte ──
# model, tokenizer = load_model_and_tokenizer(
#     "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
# )
# state = torch.load("/content/experiments/final/best_layer_weights.pt", weights_only=True)
# model_dict = dict(model.named_parameters())
# for name, tensor in state.items():
#     model_dict[name].data.copy_(tensor.to(model.device))
# print(f"Loaded {len(state)} layer parameters")

---

## Tipps für Colab

| Thema | Empfehlung |
|-------|------------|
| **GPU-Typ** | T4 (free) reicht für Qwen 0.5B. A100 (Pro) für schnellere Inferenz |
| **dtype** | Immer `float16` auf GPU → halber Speicher, schneller |
| **VRAM** | Qwen 0.5B in fp16 ≈ 1 GB. T4 hat 16 GB → viel Headroom |
| **Timeout** | Colab trennt nach ~90min Inaktivität. Halte Tab offen |
| **Ergebnisse** | Immer nach Google Drive speichern – Colab ist flüchtig! |

### Konfiguration: Voller Layer (~14.9M Parameter)

| Einstellung | Quick-Test | Experiment |
|-------------|-----------|------------|
| **components** | `"all"` | `"all"` |
| **layer_indices** | [23] | [23] |
| **σ** | 0.001 | 0.001 |
| **learning_rate** | 0.001 | 0.001 |
| **population_size** | 10 | 20 |
| **num_samples** | 15 | 25 |
| **num_generations** | 10 | 50 |
| **Inferenzen gesamt** | ~1.500 | ~25.000 |

**Wichtig bei 14.9M Dimensionen:** σ muss **sehr klein** sein (0.001), weil eine Perturbation über 14.9M Parameter sonst das Modellverhalten zerstört.